In [1]:
import re
import numpy as np
import pandas as pd
import geopandas as gpd

In [2]:
def _keep_valid_datazones(df, dz_col="Datazone"):
    """Drops the footer rows (disclosure control notes) by keeping
    only rows where the datazone column matches a real DZ code."""
    mask = df[dz_col].astype(str).str.match(r"^S00\d+$")
    return df.loc[mask].copy()

def _to_numeric_cols(df, cols):
    df = df.replace("-", 0)
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


In [3]:
def load_male_female(path= "CensusData/Male_Female_Population.csv"):
    df = pd.read_csv(path, names=["Datazone", "All People", "Female", "Male"])
    df = _keep_valid_datazones(df)
    return _to_numeric_cols(df, ["All People", "Female", "Male"])


def load_age(path= "CensusData/Age.csv"):
    df = pd.read_csv(path, skiprows=3)
    df = df.rename(columns={df.columns[0]: "Datazone"})
    df = _keep_valid_datazones(df)
    value_cols = [c for c in df.columns if c not in ("Datazone", "All people")]
    df = _to_numeric_cols(df, value_cols + ["All people"])
    return df, value_cols


def load_health(path= "CensusData/GenHealth.csv"):
    df = pd.read_csv(path, skiprows=3)
    df = df.rename(columns={df.columns[0]: "Datazone"})
    df = _keep_valid_datazones(df)
    value_cols = ["Very good", "Good", "Fair", "Bad", "Very bad"]
    df = _to_numeric_cols(df, value_cols)
    return df, value_cols


def load_cars(path= "CensusData/Car_van.csv"):
    df = pd.read_csv(path, skiprows=3)
    df = df.rename(columns={df.columns[0]: "Datazone"})
    df = _keep_valid_datazones(df)
    value_cols = [c for c in df.columns if c not in ("Datazone", "All occupied households")]
    df = _to_numeric_cols(df, value_cols)
    return df, value_cols


def load_nssec(path= "CensusData/Ns_SeC.csv"):
    df = pd.read_csv(path, skiprows=3)
    df = df.rename(columns={df.columns[0]: "Datazone"})
    df = _keep_valid_datazones(df)
    value_cols = [c for c in df.columns if c.startswith("L")]
    df = _to_numeric_cols(df, value_cols)
    return df, value_cols

def load_student(path="CensusData/studentpop.csv"):
    # expects headers: OutputArea, TotalPop, StudentPop
    df = pd.read_csv(path)
    df = df.rename(columns={"OutputArea": "Datazone", "TotalPopOver16": "TotalPop"})
    df = _keep_valid_datazones(df)
    return _to_numeric_cols(df, ["TotalPop", "StudentPop"])

def load_distance_to_work(path="CensusData/DistancetoWork.csv"):
    df = pd.read_csv(path)
    df = df.rename(columns={df.columns[0]: "Datazone", df.columns[1]: "employed_total"})
    df = df.rename(columns={
        "Mainly work from home": "wfh",
        "Less than 2km": "d_lt2",
        "2km to less than 5km": "d_2_5",
        "5km to less than 10km": "d_5_10",
        "10km to less than 20km": "d_10_20",
        "20km to less than 30km": "d_20_30",
        "30km to less than 40km": "d_30_40",
        "40km to less than 60km": "d_40_60",
        "60km and over": "d_60plus",
        "Other - No fixed place of work or working outside the UK": "no_fixed",
    })
    df = _keep_valid_datazones(df)
    value_cols = ["employed_total", "wfh", "d_lt2", "d_2_5", "d_5_10", "d_10_20",
                  "d_20_30", "d_30_40", "d_40_60", "d_60plus", "no_fixed"]
    return _to_numeric_cols(df, value_cols), value_cols


def load_pop16plus(path="CensusData/totpopover16.csv"):
    """Total population aged 16+ per Output Area (origin mass term)."""
    df = pd.read_csv(path)
    df = df.rename(columns={"OutputArea": "Datazone", "TotalPop+16": "pop_16plus"})
    df = _keep_valid_datazones(df)
    return df


def load_workplace_pop(path="CensusData/GlaWorkplacePop.csv"):
    """Workplace population per Output Area (destination mass term).
    Source: NRS Census 2022 workplace population tables."""
    df = pd.read_csv(path)
    df = df.rename(columns={"OutputCode": "Datazone", "WorkplacePop": "workplace_pop"})
    df = _keep_valid_datazones(df)
    # Strip commas from numbers like '1,379'
    df["workplace_pop"] = (
        df["workplace_pop"].astype(str).str.replace(",", "", regex=False)
    )
    df["workplace_pop"] = pd.to_numeric(df["workplace_pop"], errors="coerce").fillna(0).astype(int)
    return df


In [4]:
mf = load_male_female()
age, age_cols = load_age()
health, health_cols = load_health()
cars, cars_cols = load_cars()
nssec, nssec_cols = load_nssec()
student = load_student()
distance, distance_cols = load_distance_to_work()

for name, df in [("male_female", mf), ("age", age), ("health", health),
                  ("cars", cars), ("nssec", nssec), ("student", student),
                  ("distance", distance)]:
    print(f"{name}: {df.shape[0]} datazones, {df.shape[1]} columns")

pop16 = load_pop16plus()
workplace = load_workplace_pop()

for name, df in [("pop16plus", pop16), ("workplace", workplace)]:
    print(f"{name}: {df.shape[0]} output areas, {df.shape[1]} columns")


male_female: 46363 datazones, 4 columns
age: 46363 datazones, 103 columns
health: 46363 datazones, 7 columns
cars: 46363 datazones, 7 columns
nssec: 46363 datazones, 18 columns
student: 46363 datazones, 3 columns
distance: 46363 datazones, 12 columns
pop16plus: 46363 output areas, 2 columns
workplace: 5314 output areas, 2 columns


In [5]:
master = (
    mf[["Datazone", "Male", "Female"]]
    .merge(age[["Datazone"] + age_cols], on="Datazone")
    .merge(health[["Datazone"] + health_cols], on="Datazone")
    .merge(cars[["Datazone"] + cars_cols], on="Datazone")
    .merge(nssec[["Datazone"] + nssec_cols], on="Datazone")
    .merge(student[["Datazone", "TotalPop", "StudentPop"]], on="Datazone")
    .merge(distance[["Datazone"] + distance_cols], on="Datazone")
    .merge(pop16[["Datazone", "over16pop"]], on="Datazone", how="left")
    .merge(workplace[["Datazone", "workplace_pop"]], on="Datazone", how="left")
)

# OAs outside the workplace-pop file (non-Glasgow) get 0
master["over16pop"] = master["over16pop"].fillna(0).astype(int)
master["workplace_pop"] = master["workplace_pop"].fillna(0).astype(int)

assert master["Datazone"].is_unique
print("master shape:", master.shape)
print(f"over16pop sum: {master['over16pop'].sum():,}")
print(f"workplace_pop sum: {master['workplace_pop'].sum():,}")
master.head()


master shape: (46363, 145)
over16pop sum: 4,549,058
workplace_pop sum: 325,800


,Datazone,Male,Female,Under 1,1,2,3,4,5,6,...,d_2_5,d_5_10,d_10_20,d_20_30,d_30_40,d_40_60,d_60plus,no_fixed,over16pop,workplace_pop
0,S00135307,74,70,0,0,0,0,2,0,0,...,8,12,14,0,1,0,0,6,118,0
1,S00135308,40,27,0,4,0,1,1,0,1,...,3,5,11,0,0,1,0,11,52,0
2,S00135309,81,75,2,0,0,0,0,0,0,...,10,23,4,0,0,0,0,9,147,0
3,S00135310,68,85,2,1,1,0,0,2,0,...,13,20,8,1,0,0,0,13,132,0
4,S00135311,80,72,1,3,5,2,1,3,5,...,21,27,4,2,0,1,0,15,117,0


In [6]:
AGE_VALUE = {}
for c in age_cols:
    if c == "Under 1":
        AGE_VALUE[c] = 0
    elif c == "100 and over":
        AGE_VALUE[c] = 100
    else:
        AGE_VALUE[c] = int(c)

HEALTH_VALUE = {
    "Very good": 5, "Good": 4, "Fair": 3, "Bad": 2, "Very bad": 1,
}

CARS_VALUE = {
    "Number of cars or vans in household: No cars or vans": 0,
    "Number of cars or vans in household: One car or van": 1,
    "Number of cars or vans in household: Two cars or vans": 2,
    "Number of cars or vans in household: Three cars or vans": 3,
    "Number of cars or vans in household: Four or more cars or vans": 4,
}

NSSEC_SCORE = {}
for c in nssec_cols:
    num = int(re.match(r"L(\d+)", c).group(1))
    if num != 15:  # L15 = full-time students, excluded
        NSSEC_SCORE[c] = num

print("NS-SeC categories scored:", NSSEC_SCORE)

NS-SeC categories scored: {'L1: Employers in large establishments': 1, 'L2: Higher managerial and administrative occupations': 2, 'L3: Higher professional occupations': 3, 'L4: Lower professional and higher technical occupations': 4, 'L5: Lower managerial and administrative occupations': 5, 'L6: Higher supervisory occupations': 6, 'L7: Intermediate occupations': 7, 'L8: Employers in small establishments': 8, 'L9: Own account workers': 9, 'L10: Lower supervisory occupations': 10, 'L11: Lower technical occupations': 11, 'L12: Semi-routine occupations': 12, 'L13: Routine occupations': 13, 'L14.1: Never worked': 14, 'L14.2: Long-term unemployed': 14}


In [ ]:
def compute_station_stats(datazone_ids, master_df=master):
    sub = master_df[master_df["Datazone"].isin(datazone_ids)]
    if sub.empty:
        return {
            "n_datazones": 0, "male_female_ratio": np.nan, "avg_age": np.nan,
            "avg_health_score": np.nan, "avg_cars_per_household": np.nan,
            "avg_nssec_score": np.nan, "student_share": np.nan,
            "cycling_distance_share": np.nan, "cycling_distance_share_ext": np.nan,
            "over16pop": 0, "workplace_pop": 0,
        }

    total_male = sub["Male"].sum()
    total_female = sub["Female"].sum()
    male_female_ratio = total_male / total_female if total_female > 0 else np.nan

    age_weighted_sum = sum(sub[c].sum() * v for c, v in AGE_VALUE.items())
    age_pop = sum(sub[c].sum() for c in AGE_VALUE)
    avg_age = age_weighted_sum / age_pop if age_pop > 0 else np.nan

    health_weighted_sum = sum(sub[c].sum() * v for c, v in HEALTH_VALUE.items())
    health_pop = sum(sub[c].sum() for c in HEALTH_VALUE)
    avg_health_score = health_weighted_sum / health_pop if health_pop > 0 else np.nan

    cars_weighted_sum = sum(sub[c].sum() * v for c, v in CARS_VALUE.items())
    n_households = sum(sub[c].sum() for c in CARS_VALUE)
    avg_cars = cars_weighted_sum / n_households if n_households > 0 else np.nan

    nssec_weighted_sum = sum(sub[c].sum() * v for c, v in NSSEC_SCORE.items())
    nssec_denom = sum(sub[c].sum() for c in NSSEC_SCORE)
    avg_nssec = nssec_weighted_sum / nssec_denom if nssec_denom > 0 else np.nan

    total_pop = sub["TotalPop"].sum()
    total_students = sub["StudentPop"].sum()
    student_share = total_students / total_pop if total_pop > 0 else np.nan

    # denominator = people with an actual physical commute distance recorded (excludes
    # work from home)
    dist_band_cols = ["d_lt2", "d_2_5", "d_5_10", "d_10_20",
                       "d_20_30", "d_30_40", "d_40_60", "d_60plus"]
    commuter_pop = sum(sub[c].sum() for c in dist_band_cols)
    cycling_core = sub["d_2_5"].sum() + sub["d_5_10"].sum()               # 2-10km
    cycling_ext  = cycling_core + sub["d_10_20"].sum()                    # 2-20km, e-bike-plausible stretch
    cycling_distance_share     = cycling_core / commuter_pop if commuter_pop > 0 else np.nan
    cycling_distance_share_ext = cycling_ext  / commuter_pop if commuter_pop > 0 else np.nan

    # ── Mass terms ──
    over16pop = sub["over16pop"].sum()
    workplace_pop = sub["workplace_pop"].sum()

    return {
        "n_datazones": len(sub),
        "male_female_ratio": male_female_ratio,
        "avg_age": avg_age,
        "avg_health_score": avg_health_score,
        "avg_cars_per_household": avg_cars,
        "avg_nssec_score": avg_nssec,
        "student_share": student_share,
        "cycling_distance_share": cycling_distance_share,
        "cycling_distance_share_ext": cycling_distance_share_ext,
        "over16pop": over16pop,
        "workplace_pop": workplace_pop,
    }


# Quick sanity check against the raw data before trusting the spatial join below
print(compute_station_stats(master["Datazone"].head(5).tolist()))

{'n_datazones': 5, 'male_female_ratio': np.float64(1.0425531914893618), 'avg_age': np.float64(43.70086705202312), 'avg_health_score': np.float64(4.3546944858420265), 'avg_cars_per_household': np.float64(1.6891891891891893), 'avg_nssec_score': np.float64(7.456928838951311), 'student_share': np.float64(0.05114638447971781), 'cycling_distance_share': np.float64(0.6794258373205742), 'cycling_distance_share_ext': np.float64(0.8755980861244019), 'over16pop': np.int64(566), 'workplace_pop': np.int64(0)}


In [ ]:
STATIONS_PATH = "glasgow_stations.csv"     
DATAZONES_PATH = "CensusData/DZshp/OutputArea2022_MHW/OutputArea2022_MHW.shp"   # .shp

DZ_ID_FIELD = "code" 

BUFFER_DISTANCES_M = [150, 250, 500, 750]

stations = pd.read_csv(STATIONS_PATH)
stations = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(stations["lon"], stations["lat"]),
    crs="EPSG:4326",
)

datazones = gpd.read_file(DATAZONES_PATH)
print(datazones.columns.tolist())  # confirm DZ_ID_FIELD is correct before continuing

# Reproject both to British National Grid (metres) so the buffer distances above are metres
stations = stations.to_crs(epsg=27700)
datazones = datazones.to_crs(epsg=27700)

['code', 'HHcount', 'Popcount', 'council', 'sqkm', 'hect', 'masterpc', 'easting', 'northing', 'Shape_Leng', 'Shape_Area', 'geometry']


In [9]:
def assign_datazones_to_stations(stations_gdf, datazones_gdf, dz_id_col,
                                  buffer_distances_m, station_id_col="station_id",
                                  predicate="centroid"):
    """
    For each buffer distance, finds which datazones fall within that
    distance of each station.

    predicate='centroid'    -> a datazone counts if ITS CENTROID falls inside the
                                buffer (each datazone is assigned as a whole unit,
                                no partial/double-weighting).
    predicate='intersects'  -> a datazone counts if ANY part of its polygon
                                overlaps the buffer (more inclusive, will pull in
                                neighbouring datazones that only clip the edge).

    Returns a long dataframe: [dz_id_col, station_id_col, 'buffer_m'] —
    one row per (datazone, station, buffer distance) match.
    """
    dz_centroids = datazones_gdf.copy()
    dz_centroids["geometry"] = dz_centroids.geometry.centroid

    all_matches = []
    for buf in buffer_distances_m:
        station_buffers = stations_gdf[[station_id_col, "geometry"]].copy()
        station_buffers["geometry"] = station_buffers.geometry.buffer(buf)

        if predicate == "centroid":
            joined = gpd.sjoin(
                dz_centroids[[dz_id_col, "geometry"]], station_buffers,
                predicate="within", how="inner",
            )
        else:
            joined = gpd.sjoin(
                datazones_gdf[[dz_id_col, "geometry"]], station_buffers,
                predicate="intersects", how="inner",
            )

        joined = joined[[dz_id_col, station_id_col]].copy()
        joined["buffer_m"] = buf
        all_matches.append(joined)

    return pd.concat(all_matches, ignore_index=True)


matches = assign_datazones_to_stations(
    stations, datazones, dz_id_col=DZ_ID_FIELD,
    buffer_distances_m=BUFFER_DISTANCES_M, station_id_col="station_id",
    predicate="intersects",
)
print(matches.groupby("buffer_m")["station_id"].nunique())


buffer_m
150    121
250    121
500    121
750    121
Name: station_id, dtype: int64


In [10]:
records = []
for (station_id, buffer_m), group in matches.groupby(["station_id", "buffer_m"]):
    stats = compute_station_stats(group[DZ_ID_FIELD].tolist())
    stats.update({"station_id": station_id, "buffer_m": buffer_m})
    records.append(stats)

station_variables = pd.DataFrame(records)
station_variables = station_variables[
    ["station_id", "buffer_m", "n_datazones", "male_female_ratio", "avg_age",
     "avg_health_score", "avg_cars_per_household", "avg_nssec_score", "student_share",
     "cycling_distance_share", "cycling_distance_share_ext",
     "over16pop", "workplace_pop",]
]

station_variables.to_csv("station_independent_variables.csv", index=False)
station_variables.head(10)

,station_id,buffer_m,n_datazones,male_female_ratio,avg_age,avg_health_score,avg_cars_per_household,avg_nssec_score,student_share,cycling_distance_share,cycling_distance_share_ext,over16pop,workplace_pop
0,Alexandra Parade (West) - ELECTRIC,150,7,1.127937,33.248175,4.357753,0.756696,6.860697,0.173387,0.646388,0.790875,739,1265
1,Alexandra Parade (West) - ELECTRIC,250,17,1.088043,36.723205,4.228972,0.740552,7.825000,0.164931,0.604356,0.745917,1719,1662
2,Alexandra Parade (West) - ELECTRIC,500,66,1.033681,35.601969,4.161757,0.658824,8.345171,0.197085,0.596734,0.709055,6844,9401
3,Alexandra Parade (West) - ELECTRIC,750,139,1.026380,36.606611,4.127513,0.596500,8.513336,0.174378,0.604418,0.716867,13277,12937
4,Alexandra Park (south entrance) Alexandra Para...,150,16,0.949233,38.534010,4.125683,0.580742,7.865564,0.107583,0.667877,0.794918,1553,1066
5,Alexandra Park (south entrance) Alexandra Para...,250,33,0.970641,38.590542,4.095787,0.556500,8.151482,0.110893,0.680374,0.795327,3061,1663
6,Alexandra Park (south entrance) Alexandra Para...,500,77,1.000000,37.623414,4.096101,0.543742,8.250246,0.124603,0.651528,0.768559,6935,3708
7,Alexandra Park (south entrance) Alexandra Para...,750,128,1.020308,37.246165,4.110178,0.568294,8.263606,0.134672,0.649236,0.762939,11513,7405
8,Anderston Railway Station,150,7,1.269397,33.725073,4.250951,0.461840,7.735211,0.249738,0.303030,0.376623,957,23181
9,Anderston Railway Station,250,14,1.159898,37.841662,4.051069,0.384615,8.587816,0.188023,0.336088,0.418733,1564,23367


In [11]:
print("Census Datazone codes (master):")
print(master['Datazone'].head(10).tolist())

print("\nShapefile dzcode values (datazones):")
print(datazones['code'].head(10).tolist())

print("\nDirect overlap between the two ID sets:", 
      len(set(master['Datazone']) & set(datazones['code'])))

Census Datazone codes (master):
['S00135307', 'S00135308', 'S00135309', 'S00135310', 'S00135311', 'S00135312', 'S00135313', 'S00135314', 'S00135315', 'S00135316']

Shapefile dzcode values (datazones):
['S00135307', 'S00135308', 'S00135309', 'S00135310', 'S00135311', 'S00135312', 'S00135313', 'S00135314', 'S00135315', 'S00135316']

Direct overlap between the two ID sets: 46363


In [12]:
counts = matches.groupby(['station_id', 'buffer_m']).size().unstack('buffer_m')
print(counts.describe())

buffer_m         150         250         500         750
count     121.000000  121.000000  121.000000  121.000000
mean        8.818182   17.438017   51.975207  102.983471
std         5.180090    9.792763   23.298949   39.049324
min         1.000000    1.000000    1.000000   12.000000
25%         5.000000   10.000000   34.000000   74.000000
50%         8.000000   16.000000   47.000000   96.000000
75%        12.000000   24.000000   71.000000  129.000000
max        26.000000   42.000000  106.000000  209.000000
